# The Heat Equation — Implicit Time Stepping, Animated

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/heat.ipynb)

$\partial_t u = \Delta u$ on the unit square, integrated with backward Euler
against a manufactured multi-frequency solution. The interesting structural
point is that the time-stepping operator $M + \Delta t\,A$ is assembled and
condensed **once**: every step then rebuilds only the right-hand side, so a
hundred steps cost one factorisation, not a hundred.

The result is written out as a movie comparing the FEM solution against the
analytical one, frame by frame.

⏱️ *A couple of minutes on Colab's free CPU runtime — most of it in
rendering the movie rather than in the solve.*

Docs: [2D Heat Equation](https://docs.tensor-mesh.com/example_gallery/diffusion.html#d-heat-equation-heat-heat-py) · Source: [`examples/diffusion/heat/heat.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/diffusion/heat/heat.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# ffmpeg backs matplotlib's movie writer, which turns the time series into mp4.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa ffmpeg > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

In [ ]:
import torch

from tensormesh import Condenser, ElementAssembler, Mesh
from tensormesh.dataset import HeatMultiFrequency


class Stiffness(ElementAssembler):
    def forward(self, gradu, gradv):
        return gradu @ gradv


class MassMatrix(ElementAssembler):
    def forward(self, u, v):
        return u * v

## Assemble once, step many times

`condense_rhs` is the counterpart of the full `condenser(K, f)` call: it applies
the boundary data to a new right-hand side while reusing the condensed operator
from the first call.

In [ ]:
torch.manual_seed(3)

CHARA_LENGTH = 0.035
N_STEPS = 80
DT = 5e-5

with quiet():
    mesh = Mesh.gen_rectangle(chara_length=CHARA_LENGTH, order=2, element_type="tri")
dataset = HeatMultiFrequency(d=16)
print(f"mesh: {mesh.n_points} nodes (P2)")

M = MassMatrix.from_mesh(mesh, quadrature_order=2)()
A = Stiffness.from_mesh(mesh, quadrature_order=2)()
condenser = Condenser(mesh.boundary_mask)

# Backward Euler: (M + dt A) U^{n+1} = M U^n. The operator never changes, so it
# is condensed once and only the right-hand side is rebuilt each step.
K_ = condenser(M + DT * A)[0]

U = dataset.initial_condition(mesh.points)
Us = [U]
for _ in range(N_STEPS - 1):
    F_ = condenser.condense_rhs(M @ U)
    U = condenser.recover(K_.solve(F_))
    Us.append(U)

Us_exact = [dataset.solution(mesh.points, DT * i) for i in range(N_STEPS)]
final_err = (torch.norm(Us[-1] - Us_exact[-1]) / torch.norm(Us_exact[-1])).item()
print(f"{N_STEPS} steps done; relative L2 error at the final step: {final_err:.3e}")

## Animation

Passing a *list* of fields per panel (rather than a single tensor) is what makes
`mesh.plot` write a movie instead of a static figure.

In [ ]:
mesh.plot(
    {"FEM solution": Us, "Analytical solution": Us_exact},
    save_path="heat.mp4", dt=DT, show_mesh=False, fix_clim=False,
)
from IPython.display import Video
Video("heat.mp4", embed=True, width=780)

## Where to next

- [Wave equation](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/wave.ipynb) — second-order in time, with an energy-conservation check.
- [Allen-Cahn](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/allen_cahn.ipynb) — a nonlinear phase field, Newton solve inside every step.
- [`tensormesh.ode`](https://docs.tensor-mesh.com/api/ode.html) — ready-made explicit/implicit integrators instead of a hand-rolled loop.